In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import itertools
from collections import defaultdict
from tqdm.notebook import tqdm
from IPython.display import display, HTML

# 1. Setup
model_path = os.path.join('..', 'models', 'xgboost_wm_modelV3.joblib')
master_path = os.path.join('..', 'data', 'processed', 'features', 'MASTER_dataset.csv')

model = joblib.load(model_path)
master_df = pd.read_csv(master_path)

groups = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czech Republic'], 'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'], 'D': ['USA', 'Paraguay', 'Australia', 'Turkey'],
    'E': ['Germany', 'Curacao', 'Ivory Coast', 'Ecuador'], 'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'], 'H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
    'I': ['France', 'Senegal', 'Iraq', 'Norway'], 'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'], 'L': ['England', 'Croatia', 'Ghana', 'Panama']
}

features = ['Delta_Total_Market_Value', 'Delta_Median_Top11_Value', 'Delta_Chemistry', 'Delta_Form_Rating', 'Delta_UCL_Minutes', 'Delta_Tournament_Minutes', 'Delta_Average_Age', 'Delta_TM_Value_Rank', 'Delta_FIFA_Rank', 'Delta_FIFA_Points', 'Is_Neutral']

# NEU: Wir packen ALLE 48 Teams in eine flache Liste
all_teams = [team for group_teams in groups.values() for team in group_teams]

print("🧠 Berechne Basis-Wahrscheinlichkeiten für ALLE 1.128 möglichen WM-Spiele...")
match_probs = {}
team_fifa_points = {}

# FIFA Punkte für alle Teams holen
for team in all_teams:
    team_data = master_df[master_df['Country'].str.lower() == team.lower()]
    team_fifa_points[team] = team_data.iloc[0]['FIFA_Points']

# NEU: Wir berechnen die Wahrscheinlichkeiten für JEDE mögliche Paarung (wichtig für die K.o.-Runde)
for team_a, team_b in itertools.combinations(all_teams, 2):
    data_a = master_df[master_df['Country'].str.lower() == team_a.lower()].iloc[0]
    data_b = master_df[master_df['Country'].str.lower() == team_b.lower()].iloc[0]
    
    delta_dict = {
        'Delta_Total_Market_Value': data_a['Total_Market_Value_mEUR'] - data_b['Total_Market_Value_mEUR'],
        'Delta_Median_Top11_Value': data_a['Median_Top11_Market_Value_mEUR'] - data_b['Median_Top11_Market_Value_mEUR'],
        'Delta_Chemistry': data_a['Chemistry_Score'] - data_b['Chemistry_Score'],
        'Delta_Form_Rating': data_a['Current_Form_Rating'] - data_b['Current_Form_Rating'],
        'Delta_UCL_Minutes': data_a['Total_UCL_Minutes'] - data_b['Total_UCL_Minutes'],
        'Delta_Tournament_Minutes': data_a['Total_Tournament_Minutes'] - data_b['Total_Tournament_Minutes'],
        'Delta_Average_Age': data_a['Average_Age'] - data_b['Average_Age'],
        'Delta_TM_Value_Rank': data_a['TM_Value_Rank'] - data_b['TM_Value_Rank'],
        'Delta_FIFA_Rank': data_a['FIFA_Rank'] - data_b['FIFA_Rank'],
        'Delta_FIFA_Points': data_a['FIFA_Points'] - data_b['FIFA_Points'],
        'Is_Neutral': 1
    }
    X_pred = pd.DataFrame([delta_dict])[features]
    probs = model.predict_proba(X_pred)[0]
    match_probs[(team_a, team_b)] = probs

# Wunderschöne Ausgabe für das Notebook (aber nur für die Gruppenphase, sonst wird es zu lang)
display(HTML("<h2>🔍 Wahrscheinlichkeiten der Gruppenspiele</h2>"))

for group_name, teams in groups.items():
    group_match_data = []
    matchups = list(itertools.combinations(teams, 2))
    
    for team_a, team_b in matchups:
        # Hier greifen wir jetzt einfach auf unseren vollen Zwischenspeicher zu
        probs = match_probs.get((team_a, team_b), match_probs.get((team_b, team_a)))
        
        # Falls die Reihenfolge andersrum im Dictionary liegt, drehen wir die Ausgabe um
        if (team_b, team_a) in match_probs:
            probs = [probs[2], probs[1], probs[0]]
            
        group_match_data.append({
            'Match': f"{team_a} vs {team_b}",
            f'Sieg {team_a}': f"{probs[2]*100:.1f}%",
            'Remis': f"{probs[1]*100:.1f}%",
            f'Sieg {team_b}': f"{probs[0]*100:.1f}%"
        })
        
    display(HTML(f"<h4>Gruppe {group_name}</h4>"))
    df_group_probs = pd.DataFrame(group_match_data)
    display(df_group_probs.style.hide(axis="index").set_properties(**{'text-align': 'center'}))

🧠 Berechne Basis-Wahrscheinlichkeiten für ALLE 1.128 möglichen WM-Spiele...


Match,Sieg Mexico,Remis,Sieg South Africa,Sieg South Korea,Sieg Czech Republic
Mexico vs South Africa,51.3%,17.4%,31.3%,nan,nan
Mexico vs South Korea,35.8%,23.4%,nan,40.8%,nan
Mexico vs Czech Republic,32.8%,18.8%,nan,nan,48.4%
South Africa vs South Korea,nan,19.3%,30.6%,50.1%,nan
South Africa vs Czech Republic,nan,16.7%,30.8%,nan,52.5%
South Korea vs Czech Republic,nan,28.4%,nan,30.0%,41.6%


Match,Sieg Canada,Remis,Sieg Bosnia and Herzegovina,Sieg Qatar,Sieg Switzerland
Canada vs Bosnia and Herzegovina,55.8%,14.6%,29.6%,nan,nan
Canada vs Qatar,51.1%,22.0%,nan,26.9%,nan
Canada vs Switzerland,22.9%,24.1%,nan,nan,53.0%
Bosnia and Herzegovina vs Qatar,nan,22.9%,37.6%,39.4%,nan
Bosnia and Herzegovina vs Switzerland,nan,16.8%,28.6%,nan,54.6%
Qatar vs Switzerland,nan,18.8%,nan,19.2%,62.0%


Match,Sieg Brazil,Remis,Sieg Morocco,Sieg Haiti,Sieg Scotland
Brazil vs Morocco,45.1%,26.0%,28.9%,nan,nan
Brazil vs Haiti,57.3%,16.0%,nan,26.7%,nan
Brazil vs Scotland,34.7%,23.4%,nan,nan,42.0%
Morocco vs Haiti,nan,12.6%,63.1%,24.3%,nan
Morocco vs Scotland,nan,21.7%,36.7%,nan,41.6%
Haiti vs Scotland,nan,18.9%,nan,24.9%,56.2%


Match,Sieg USA,Remis,Sieg Paraguay,Sieg Australia,Sieg Turkey
USA vs Paraguay,54.4%,22.7%,22.9%,nan,nan
USA vs Australia,54.8%,22.4%,nan,22.8%,nan
USA vs Turkey,33.5%,14.5%,nan,nan,52.1%
Paraguay vs Australia,nan,33.4%,31.5%,35.1%,nan
Paraguay vs Turkey,nan,14.6%,36.4%,nan,49.0%
Australia vs Turkey,nan,16.4%,nan,36.4%,47.2%


...

Match,Sieg Spain,Remis,Sieg Cape Verde,Sieg Saudi Arabia,Sieg Uruguay
Spain vs Cape Verde,40.1%,18.7%,41.3%,nan,nan
Spain vs Saudi Arabia,47.1%,15.1%,nan,37.8%,nan
Spain vs Uruguay,51.2%,23.3%,nan,nan,25.5%
Cape Verde vs Saudi Arabia,nan,27.9%,27.2%,44.8%,nan
Cape Verde vs Uruguay,nan,19.7%,22.0%,nan,58.3%
Saudi Arabia vs Uruguay,nan,17.4%,nan,20.3%,62.3%


Match,Sieg France,Remis,Sieg Senegal,Sieg Iraq,Sieg Norway
France vs Senegal,45.2%,17.5%,37.3%,nan,nan
France vs Iraq,49.3%,13.6%,nan,37.1%,nan
France vs Norway,52.1%,19.2%,nan,nan,28.7%
Senegal vs Iraq,nan,13.0%,63.5%,23.5%,nan
Senegal vs Norway,nan,28.2%,43.8%,nan,28.0%
Iraq vs Norway,nan,22.1%,nan,21.7%,56.3%


Match,Sieg Argentina,Remis,Sieg Algeria,Sieg Austria,Sieg Jordan
Argentina vs Algeria,49.2%,23.6%,27.1%,nan,nan
Argentina vs Austria,53.4%,23.0%,nan,23.6%,nan
Argentina vs Jordan,58.0%,12.3%,nan,nan,29.7%
Algeria vs Austria,nan,25.8%,26.0%,48.2%,nan
Algeria vs Jordan,nan,21.0%,55.3%,nan,23.7%
Austria vs Jordan,nan,21.4%,nan,56.4%,22.2%


Match,Sieg Portugal,Remis,Sieg DR Congo,Sieg Uzbekistan,Sieg Colombia
Portugal vs DR Congo,57.9%,13.1%,29.0%,nan,nan
Portugal vs Uzbekistan,53.8%,12.5%,nan,33.6%,nan
Portugal vs Colombia,40.3%,19.9%,nan,nan,39.8%
DR Congo vs Uzbekistan,nan,53.9%,27.9%,18.2%,nan
DR Congo vs Colombia,nan,26.2%,23.5%,nan,50.4%
Uzbekistan vs Colombia,nan,18.5%,nan,20.9%,60.7%


Match,Sieg England,Remis,Sieg Croatia,Sieg Ghana,Sieg Panama
England vs Croatia,43.2%,17.5%,39.3%,nan,nan
England vs Ghana,49.8%,15.0%,nan,35.2%,nan
England vs Panama,41.9%,26.2%,nan,nan,31.9%
Croatia vs Ghana,nan,14.9%,55.4%,29.8%,nan
Croatia vs Panama,nan,20.3%,59.5%,nan,20.3%
Ghana vs Panama,nan,26.0%,nan,31.1%,42.8%


In [5]:
# Hilfsfunktion: Berechnet ein Spiel OHNE Unentschieden (Wahrscheinlichkeit wird umverteilt)
def play_ko_match(team_a, team_b):
    if (team_a, team_b) in match_probs:
        probs = match_probs[(team_a, team_b)]
        p_a = probs[2] / (probs[2] + probs[0]) # Nur Sieg A und Sieg B vergleichen
        return team_a if np.random.rand() < p_a else team_b
    else:
        probs = match_probs[(team_b, team_a)]
        p_b = probs[2] / (probs[2] + probs[0])
        return team_b if np.random.rand() < p_b else team_a

def play_round(teams):
    return [play_ko_match(teams[i], teams[i+1]) for i in range(0, len(teams), 2)]

In [ ]:
simulations = 10000

# Tracking für die Gruppenphase
group_stats = defaultdict(lambda: {'1st': 0, '2nd': 0, '3rd': 0, '4th': 0, 'Advances_Group': 0})
# Tracking für die K.o.-Phase
ko_stats = defaultdict(lambda: {'R32': 0, 'R16': 0, 'QF': 0, 'SF': 0, 'Final': 0, 'Win': 0})

for _ in tqdm(range(simulations), desc="Simuliere komplette Weltmeisterschaften"):
    winners, runners_up, thirds = {}, {}, []
    
    # --- 1. GRUPPENPHASE ---
    for group_name, teams in groups.items():
        standings = {team: 0 for team in teams}
        for team_a, team_b in itertools.combinations(teams, 2):
            probs = match_probs[(team_a, team_b)]
            outcome = np.random.choice([0, 1, 2], p=probs)
            if outcome == 2: standings[team_a] += 3
            elif outcome == 0: standings[team_b] += 3
            else:
                standings[team_a] += 1
                standings[team_b] += 1
                
        sorted_group = sorted(standings.items(), key=lambda x: (x[1], team_fifa_points[x[0]]), reverse=True)
        
        t1, t2, t3, t4 = [t[0] for t in sorted_group]
        
        # Tracking für die End-Tabellen
        group_stats[t1]['1st'] += 1
        group_stats[t2]['2nd'] += 1
        group_stats[t3]['3rd'] += 1
        group_stats[t4]['4th'] += 1
        
        group_stats[t1]['Advances_Group'] += 1
        group_stats[t2]['Advances_Group'] += 1
        
        # Plätze speichern für den K.o.-Baum
        winners[group_name] = t1
        runners_up[group_name] = t2
        thirds.append((t3, sorted_group[2][1], team_fifa_points[t3]))
        
    # Die 8 besten Dritten ermitteln
    best_thirds = [t[0] for t in sorted(thirds, key=lambda x: (x[1], x[2]), reverse=True)[:8]]
    for t3_team in best_thirds:
        group_stats[t3_team]['Advances_Group'] += 1
    
    # --- 2. TURNIER-BAUM (Sechzehntelfinale) ---
    r32_teams = [
        winners['A'], best_thirds[0], winners['B'], best_thirds[1],
        winners['C'], best_thirds[2], winners['D'], best_thirds[3],
        winners['E'], best_thirds[4], winners['F'], best_thirds[5],
        winners['G'], best_thirds[6], winners['H'], best_thirds[7],
        winners['I'], runners_up['A'], winners['J'], runners_up['B'],
        winners['K'], runners_up['C'], winners['L'], runners_up['D'],
        runners_up['E'], runners_up['F'], runners_up['G'], runners_up['H'],
        runners_up['I'], runners_up['J'], runners_up['K'], runners_up['L']
    ]
    for t in r32_teams: ko_stats[t]['R32'] += 1
        
    # --- 3. K.O.-PHASE SPIELEN ---
    r16_teams = play_round(r32_teams)
    for t in r16_teams: ko_stats[t]['R16'] += 1
        
    qf_teams = play_round(r16_teams)
    for t in qf_teams: ko_stats[t]['QF'] += 1
        
    sf_teams = play_round(qf_teams)
    for t in sf_teams: ko_stats[t]['SF'] += 1
        
    final_teams = play_round(sf_teams)
    for t in final_teams: ko_stats[t]['Final'] += 1
        
    champion = play_round(final_teams)[0]
    ko_stats[champion]['Win'] += 1

# ==========================================
# AUSGABE 1: WUNDERSCHÖNE GRUPPENTABELLEN
# ==========================================
display(HTML("<h2>📊 GRUPPENPHASEN-ÜBERSICHT</h2>"))
for group_name, teams in groups.items():
    display(HTML(f"<h4>--- GRUPPE {group_name} ---</h4>"))
    group_data = []
    for team in teams:
        t_stats = group_stats[team]
        group_data.append({
            'Team': team,
            '1. Platz': f"{(t_stats['1st'] / simulations) * 100:.1f}%",
            '2. Platz': f"{(t_stats['2nd'] / simulations) * 100:.1f}%",
            '3. Platz': f"{(t_stats['3rd'] / simulations) * 100:.1f}%",
            '4. Platz': f"{(t_stats['4th'] / simulations) * 100:.1f}%",
            'WEITERKOMMEN (Top 2 oder beste Dritte)': f"{(t_stats['Advances_Group'] / simulations) * 100:.1f}%"
        })
    df_out = pd.DataFrame(group_data)
    df_out = df_out.iloc[df_out['WEITERKOMMEN (Top 2 oder beste Dritte)'].str.rstrip('%').astype(float).argsort()[::-1]]
    display(df_out.style.hide(axis="index").set_properties(**{'text-align': 'center'}))

# ==========================================
# AUSGABE 2: FINALE WM-PROGNOSE (K.o.-Runde)
# ==========================================
display(HTML("<h2>🏆 DIE WM-PROGNOSE (Nach 10.000 Simulationen)</h2>"))
results = []
for team, t_stats in ko_stats.items():
    results.append({
        'Team': team,
        'Sechzehntelfinale': (t_stats['R32'] / simulations) * 100,
        'Achtelfinale': (t_stats['R16'] / simulations) * 100,
        'Viertelfinale': (t_stats['QF'] / simulations) * 100,
        'Halbfinale': (t_stats['SF'] / simulations) * 100,
        'Finale': (t_stats['Final'] / simulations) * 100,
        'WELTMEISTER': (t_stats['Win'] / simulations) * 100
    })

df_results = pd.DataFrame(results).sort_values(by='WELTMEISTER', ascending=False).reset_index(drop=True)
format_dict = {col: '{:.1f}%' for col in df_results.columns if col != 'Team'}
display(df_results.style.format(format_dict).set_properties(**{'text-align': 'center'}).background_gradient(subset=['WELTMEISTER'], cmap='Greens'))

Simuliere komplette Weltmeisterschaften:   0%|          | 0/10000 [00:00<?, ?it/s]

,Team,Sechzehntelfinale,Achtelfinale,Viertelfinale,Halbfinale,Finale,WELTMEISTER
0,Argentina,89.1%,58.7%,35.6%,21.8%,13.5%,8.4%
1,Spain,82.3%,53.5%,33.6%,20.5%,12.6%,7.9%
2,France,84.7%,50.7%,30.8%,18.4%,10.6%,6.4%
3,Portugal,85.4%,50.0%,29.0%,17.3%,10.1%,5.9%
4,Netherlands,87.9%,52.8%,31.3%,17.6%,10.4%,5.7%
5,Brazil,84.5%,52.0%,32.9%,19.9%,10.7%,5.6%
6,Morocco,79.4%,49.8%,29.9%,17.6%,9.9%,5.3%
7,Colombia,85.5%,46.5%,26.4%,15.2%,8.1%,4.4%
8,Belgium,87.2%,50.5%,27.9%,15.3%,8.7%,4.4%
9,Senegal,82.7%,48.7%,26.6%,14.4%,7.8%,4.1%
